在 Python 中，signal 和 sys 是两个非常核心且经常配合使用的标准库模块。
- signal 模块：用于处理操作系统信号（Signals）。它允许你设置回调函数，当程序接收到特定信号（如用户按下 Ctrl+C、进程被终止、定时器到期等）时执行自定义逻辑。

- sys 模块：提供了访问与 Python 解释器 紧密相关的变量和函数。常用于获取命令行参数、控制程序退出、操作标准输入输出流等。

它们最常见的配合场景是：捕获信号（如中断），进行清理工作，然后优雅地退出程序。

### signal模块：处理异步事件

信号是一种软件中断。当进程收到信号时，正常的执行流会被打断，转而去执行你注册的“信号处理函数”。

<b>常用信号常量</b>
- signal.SIGINT: 对应键盘中断（通常是 Ctrl+C）。
- signal.SIGTERM: 对应终止请求（kill 命令默认发送的信号）。
- signal.SIGALRM: 闹钟信号（仅在 Unix/Linux 上可用，用于定时器）。
- signal.SIGUSR1, signal.SIGUSR2: 用户自定义信号。


#### 优雅地处理 Ctrl+C

如果不处理 SIGINT，按下 Ctrl+C 会直接抛出 KeyboardInterrupt 异常并可能导致程序崩溃或资源未释放。使用 signal 可以捕获它

In [ ]:
import signal
import sys
import time

# 定义信号处理函数
# 必须接受两个参数，sigum（信号编号） 和frame（当前栈帧）
def handle_interrupt(signum,frame):
    print(f"\n[警告]接收到信号{signum}（通常是ctrl+c)")
    print('正在保存进度并清理资源')
    
    # 在这里执行清理逻辑，比如关闭文件，断开数据库连接
    time.sleep(1) # 模拟清理耗时
    
    print('清理完毕，程序即将退出')
    # 使用sys.exit(0)优雅退出
    sys.exit(0)

# 注册信号处理函数
# 告诉python：当收到sigint信号时，调用handle_interrupt函数
signal.signal(signal.SIGINT,handle_interrupt)

print('程序运行中... 请按ctrl+c 测试终端处理')


try:
    while True:
        print('.',end='',flush=True)
        time.sleep(1)
except Exception as e:
    print(f'发生未知错误：{e}')

### sys 模块：系统交互与控制

sys 模块提供了更底层的控制能力，常与 signal 配合使用来控制程序的生命周期。

1. sys.exit([arg]): 退出程序。
- 这是 signal 处理函数中最常用的结束方式。
- arg 为 0 表示正常退出，非 0 表示异常退出。
2. sys.argv: 获取命令行参数列表。
- sys.argv[0] 是脚本名称，后续是传入的参数。
3. sys.stdin, sys.stdout, sys.stderr: 标准输入/输出/错误流。
4. sys.path: Python 搜索模块的路径列表

#### 配合 signal 的退出逻辑
在信号处理中，我们通常不希望程序直接崩溃，而是希望执行完清理代码后，通过 sys.exit() 告诉操作系统程序已正常结束。

In [ ]:
import sys

def cleanup_and_exit(signum,frame):
    print("\n清理资源...")
    # 模拟关闭数据库，删除临时文件等操作
    
    # 决定退出吗
    if signum == signal.SIGINT:
        code = 0 # 用户手动中断，视为正常退出
    else:
        code = 1 # 其他信号，视为异常
    print(f'程序退出，状态码：{code}')
    sys.exit(code) # 关键：再次处退出

### 高级场景：超时控制

利用signal.SIGALRM 和sys可以实现函数的超时强制退出，这在防止网络请求卡死或死循环时非常有用

注意： SIGALRM 仅在 Unix/Linux/macOS 上有效，Windows 不支持。

In [ ]:
class TimeoutError(Exception):
    '''
    创建一个专门的异常类，用于标记’超时‘这一特定错误
    '''
    pass 

def timeout_handler(signum,frame):
    # 当超时时，抛出一个异常而不是直接退出，这样可以在try-except中捕获
    raise TimeoutError

def run_with_timeout(func,timeout_seconds,*args,**kwargs):
    # 注册SIGALARM的处理函数
    old_handler = signal.signal(signal.SIGALARM,timeout_handler)
    # 设置闹钟，timeout_seconds秒触发SIGALRM
    signal.alarm(timeout_seconds)
    
    try:
        retult = func(*args,**kwwargs)
        # 如果函数在超时完成，取消闹钟
        signal.alarm(0)
        return result
    except TimeoutError as e:
        print(f"捕获到超时错误:{e}")
        return None
    finally:
        # 恢复旧的信号处理函数
        signal.signal(signal.SIGALRM,old_handler)
        
# 测试函数
def slow_task():
    print('开始处理耗时任务...')
    time.sleep(5) # 模拟耗时5秒
    print('任务处理完成')
    return '成功'

if __name__ == '__main__':
    print('尝试运行一个需要5秒的任务，但限制时间为2秒...')
    result = run_with_timeout(slow_task,timeout_seconds=2)
    
    if result is None:
        print('任务未能处理按时处理完成')
    else:
        print(f'任务结果：{result}')